# Idiosyncratic Volatility Research

This notebook prototypes and validates a rolling market-model residual-volatility factor before it is integrated into the production factor pipeline.

### Cell 1 — Load and audit the data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Work whether the notebook kernel starts from the repository root
# or from the notebooks directory.
current = Path.cwd().resolve()
for _ in range(10):
    if (current / "pyproject.toml").exists():
        ROOT = current
        break
    current = current.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

equity_path = ROOT / "data" / "processed" / "equity_panel.parquet"
spy_path = ROOT / "data" / "raw" / "spy_benchmark.parquet"

equity = pd.read_parquet(equity_path)
spy = pd.read_parquet(spy_path)

equity["date"] = pd.to_datetime(equity["date"])
spy["date"] = pd.to_datetime(spy["date"])

if "ticker" in spy.columns:
    spy = spy.loc[spy["ticker"] == "SPY"].copy()

required_equity = {"date", "ticker", "adj_close", "ret_1d"}
required_spy = {"date", "adj_close"}

assert required_equity.issubset(equity.columns)
assert required_spy.issubset(spy.columns)

print("Project root:", ROOT)
print("Equity shape:", equity.shape)
print("Equity tickers:", equity["ticker"].nunique())
print("Equity period:", equity["date"].min(), "to", equity["date"].max())
print("Duplicate equity date/ticker rows:",
      equity.duplicated(["date", "ticker"]).sum())

print()
print("SPY shape:", spy.shape)
print("SPY period:", spy["date"].min(), "to", spy["date"].max())
print("Duplicate SPY dates:", spy.duplicated(["date"]).sum())

Project root: D:\dev\quant_projects\alpha-research-lab
Equity shape: (284249, 18)
Equity tickers: 101
Equity period: 2015-01-02 00:00:00 to 2026-07-02 00:00:00
Duplicate equity date/ticker rows: 0

SPY shape: (2891, 8)
SPY period: 2015-01-02 00:00:00 to 2026-07-02 00:00:00
Duplicate SPY dates: 0


### Cell 2 — Construct aligned stock and market returns

In [6]:
market = (
    spy[["date", "adj_close"]]
    .sort_values("date")
    .drop_duplicates("date")
    .rename(columns={"adj_close": "market_adj_close"})
    .reset_index(drop=True)
)

# This is the backward-looking return from t-1 to t,
# matching equity_panel["ret_1d"].
market["market_ret_1d"] = market["market_adj_close"].pct_change()

aligned = equity[["date", "ticker", "ret_1d"]].merge(
    market[["date", "market_ret_1d"]],
    on="date",
    how="left",
    validate="many_to_one",
)

aligned_counts = (
    aligned.dropna(subset=["ret_1d", "market_ret_1d"])
    .groupby("ticker")
    .size()
    .sort_values(ascending=False)
)

print(aligned_counts.describe())
print()
print("Five tickers with most aligned observations:")
print(aligned_counts.head())
print()
print("Five tickers with fewest aligned observations:")
print(aligned_counts.tail())

count     101.000000
mean     2813.346535
std       404.669036
min        12.000000
25%      2890.000000
50%      2890.000000
75%      2890.000000
max      2890.000000
dtype: float64

Five tickers with most aligned observations:
ticker
AAPL    2890
ABBV    2890
ABT     2890
ACN     2890
ADBE    2890
dtype: int64

Five tickers with fewest aligned observations:
ticker
V       2890
UBER    1795
PLTR    1444
GEV      567
HONA      12
dtype: int64


### Cell 3 — Manually estimate one 63-day regression

In [5]:
WINDOW = 63
ANNUALISATION_FACTOR = 252

sample_ticker = aligned_counts.index[0]

sample = (
    aligned.loc[aligned["ticker"] == sample_ticker]
    .dropna(subset=["ret_1d", "market_ret_1d"])
    .sort_values("date")
    .reset_index(drop=True)
)

window_data = sample.tail(WINDOW).copy()

assert len(window_data) == WINDOW

y = window_data["ret_1d"].to_numpy()
market_returns = window_data["market_ret_1d"].to_numpy()

# The first column estimates the intercept alpha.
X = np.column_stack(
    [
        np.ones(WINDOW),
        market_returns,
    ]
)

coefficients, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
alpha, beta = coefficients

residuals = y - X @ coefficients

idio_vol = residuals.std(ddof=1) * np.sqrt(ANNUALISATION_FACTOR)
total_vol = y.std(ddof=1) * np.sqrt(ANNUALISATION_FACTOR)

sse = np.sum(residuals**2)
sst = np.sum((y - y.mean()) ** 2)
r_squared = 1.0 - sse / sst

print("Ticker:", sample_ticker)
print("Window:", window_data["date"].min(), "to", window_data["date"].max())
print("Observations:", len(window_data))
print(f"Daily alpha: {alpha:.8f}")
print(f"Beta: {beta:.6f}")
print(f"Annualised total volatility: {total_vol:.4%}")
print(f"Annualised idiosyncratic volatility: {idio_vol:.4%}")
print(f"R-squared: {r_squared:.4f}")
print(f"Residual mean: {residuals.mean():.3e}")

Ticker: AAPL
Window: 2026-04-02 00:00:00 to 2026-07-02 00:00:00
Observations: 63
Daily alpha: 0.00200195
Beta: 0.539998
Annualised total volatility: 28.1303%
Annualised idiosyncratic volatility: 27.1284%
R-squared: 0.0700
Residual mean: -1.322e-18


### Cell 4 — Reproduce the manual result with rolling formulas

In [7]:
rolling_stock = sample["ret_1d"].rolling(
    WINDOW,
    min_periods=WINDOW,
)
rolling_market = sample["market_ret_1d"].rolling(
    WINDOW,
    min_periods=WINDOW,
)

stock_mean = rolling_stock.mean()
market_mean = rolling_market.mean()

stock_variance = rolling_stock.var(ddof=1)
market_variance = rolling_market.var(ddof=1)
covariance = rolling_stock.cov(sample["market_ret_1d"])

rolling_beta = covariance / market_variance
rolling_alpha = stock_mean - rolling_beta * market_mean

# Equivalent to the sample variance of the in-window OLS residuals.
residual_variance = (
    stock_variance
    - covariance.pow(2) / market_variance
).clip(lower=0.0)

rolling_idio_vol = (
    np.sqrt(residual_variance)
    * np.sqrt(ANNUALISATION_FACTOR)
)

comparison = pd.Series(
    {
        "manual_alpha": alpha,
        "rolling_alpha": rolling_alpha.iloc[-1],
        "manual_beta": beta,
        "rolling_beta": rolling_beta.iloc[-1],
        "manual_idio_vol": idio_vol,
        "rolling_idio_vol": rolling_idio_vol.iloc[-1],
    }
)

print(comparison)
print()
print(
    "Alpha agrees:",
    np.isclose(alpha, rolling_alpha.iloc[-1]),
)
print(
    "Beta agrees:",
    np.isclose(beta, rolling_beta.iloc[-1]),
)
print(
    "Idiosyncratic volatility agrees:",
    np.isclose(idio_vol, rolling_idio_vol.iloc[-1]),
)

manual_alpha        0.002002
rolling_alpha       0.002002
manual_beta         0.539998
rolling_beta        0.539998
manual_idio_vol     0.271284
rolling_idio_vol    0.271284
dtype: float64

Alpha agrees: True
Beta agrees: True
Idiosyncratic volatility agrees: True


### Cell 5 — Apply the rolling calculation across the universe

In [8]:
def rolling_market_model_for_ticker(
    group: pd.DataFrame,
    window: int = 63,
    annualisation_factor: int = 252,
) -> pd.DataFrame:
    """Prototype a rolling market model for one ticker."""
    group = group.sort_values("date").copy()

    stock_return = group["ret_1d"]
    market_return = group["market_ret_1d"]

    # Ensure all rolling statistics use the same paired observations.
    paired_stock = stock_return.where(market_return.notna())
    paired_market = market_return.where(stock_return.notna())

    rolling_stock = paired_stock.rolling(
        window,
        min_periods=window,
    )
    rolling_market = paired_market.rolling(
        window,
        min_periods=window,
    )

    stock_mean = rolling_stock.mean()
    market_mean = rolling_market.mean()

    stock_variance = rolling_stock.var(ddof=1)
    market_variance = rolling_market.var(ddof=1)
    covariance = rolling_stock.cov(paired_market)

    valid_market_variance = market_variance > 0

    group["alpha_63"] = (
        stock_mean - covariance / market_variance * market_mean
    ).where(valid_market_variance)

    group["beta_63"] = (
        covariance / market_variance
    ).where(valid_market_variance)

    residual_variance = (
        stock_variance
        - covariance.pow(2) / market_variance
    ).clip(lower=0.0)

    group["idio_vol_63_raw"] = (
        np.sqrt(residual_variance)
        * np.sqrt(annualisation_factor)
    ).where(valid_market_variance)

    return group


rolling_results = pd.concat(
    [
        rolling_market_model_for_ticker(group, WINDOW)
        for _, group in aligned.groupby("ticker", sort=False)
    ],
    ignore_index=True,
)

coverage_by_ticker = (
    rolling_results.groupby("ticker")["idio_vol_63_raw"]
    .agg(["count", "min", "median", "max"])
    .sort_values("count")
)

latest_cross_section = (
    rolling_results.dropna(subset=["idio_vol_63_raw"])
    .sort_values(["ticker", "date"])
    .groupby("ticker")
    .tail(1)
)

print("Rows:", len(rolling_results))
print(
    "Non-missing idiosyncratic volatility:",
    rolling_results["idio_vol_63_raw"].notna().sum(),
)
print(
    "Tickers with at least one estimate:",
    rolling_results.loc[
        rolling_results["idio_vol_63_raw"].notna(),
        "ticker",
    ].nunique(),
)
print()
print("Lowest coverage:")
print(coverage_by_ticker.head())
print()
print("Latest cross-sectional distribution:")
print(latest_cross_section["idio_vol_63_raw"].describe())
print()
print("Any negative estimates:")
print((rolling_results["idio_vol_63_raw"] < 0).any())

Rows: 284249
Non-missing idiosyncratic volatility: 277936
Tickers with at least one estimate: 100

Lowest coverage:
        count       min    median       max
ticker                                     
HONA        0       NaN       NaN       NaN
GEV       505  0.293517  0.404640  0.611693
PLTR     1382  0.326664  0.518116  1.255472
UBER     1733  0.217724  0.386131  1.070705
ADBE     2828  0.093065  0.219365  0.479237

Latest cross-sectional distribution:
count    100.000000
mean       0.333682
std        0.145951
min        0.128565
25%        0.240955
50%        0.288401
75%        0.367795
max        0.853370
Name: idio_vol_63_raw, dtype: float64

Any negative estimates:
False


### Cell 6 — Verify the new feature generated from script

In [9]:
factor_panel = pd.read_parquet(
    ROOT / "data" / "processed" / "factor_panel.parquet"
)

expected_columns = [
    "idio_vol_63_raw",
    "idio_vol_63_winsorised",
    "idio_vol_63_z",
    "idio_vol_63_rank",
    "idio_vol_63_sector_neutral_z",
    "beta_126",
]

print("Factor-panel shape:", factor_panel.shape)
print("Expected columns present:")
print(pd.Series({column: column in factor_panel for column in expected_columns}))

intermediate_columns = [
    column
    for column in factor_panel.columns
    if column.startswith("market_model_63_")
]

print()
print("Retained intermediate model columns:", intermediate_columns)

print()
print("Idiosyncratic-volatility coverage:")
print(
    factor_panel["idio_vol_63_raw"]
    .agg(["count", "min", "median", "max"])
)

print()
print("Coverage by ticker, lowest five:")
print(
    factor_panel.groupby("ticker")["idio_vol_63_raw"]
    .count()
    .sort_values()
    .head()
)

assert set(expected_columns).issubset(factor_panel.columns)
assert intermediate_columns == []
assert not (factor_panel["idio_vol_63_raw"].dropna() < 0).any()

Factor-panel shape: (284249, 44)
Expected columns present:
idio_vol_63_raw                 True
idio_vol_63_winsorised          True
idio_vol_63_z                   True
idio_vol_63_rank                True
idio_vol_63_sector_neutral_z    True
beta_126                        True
dtype: bool

Retained intermediate model columns: []

Idiosyncratic-volatility coverage:
count     277936.000000
min            0.058685
median         0.195293
max            1.255472
Name: idio_vol_63_raw, dtype: float64

Coverage by ticker, lowest five:
ticker
HONA       0
GEV      505
PLTR    1382
UBER    1733
ADBE    2828
Name: idio_vol_63_raw, dtype: int64


In [10]:
pipeline_comparison = factor_panel[
    ["date", "ticker", "idio_vol_63_raw"]
].merge(
    rolling_results[
        ["date", "ticker", "idio_vol_63_raw"]
    ].rename(
        columns={
            "idio_vol_63_raw": "prototype_idio_vol_63_raw",
        }
    ),
    on=["date", "ticker"],
    how="outer",
    validate="one_to_one",
)

maximum_difference = (
    pipeline_comparison["idio_vol_63_raw"]
    - pipeline_comparison["prototype_idio_vol_63_raw"]
).abs().max()

all_values_agree = np.allclose(
    pipeline_comparison["idio_vol_63_raw"],
    pipeline_comparison["prototype_idio_vol_63_raw"],
    equal_nan=True,
)

print("Maximum absolute difference:", maximum_difference)
print("All pipeline and prototype values agree:", all_values_agree)

assert all_values_agree

Maximum absolute difference: 0.0
All pipeline and prototype values agree: True


## Robustness analysis

### IC by horizon and signal treatment

In [11]:
from alpha_research.validation import (
    calculate_ic_by_horizon,
    calculate_quantile_returns,
    calculate_subperiod_ic,
)

signal_definitions = {
    "Raw": "idio_vol_63_z",
    "Sector neutral": "idio_vol_63_sector_neutral_z",
}

horizon_results = []

for signal_name, factor_column in signal_definitions.items():
    result = calculate_ic_by_horizon(
        factor_panel,
        factor_column=factor_column,
        forward_return_columns=[
            "forward_ret_1d",
            "forward_ret_5d",
        ],
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    horizon_results.append(result)

idio_horizon_summary = pd.concat(
    horizon_results,
    ignore_index=True,
)

idio_horizon_summary[
    [
        "signal",
        "forward_return_column",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,forward_return_column,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,Raw,forward_ret_1d,2827.0,0.003546,0.214868,0.016504,0.877504,0.513972
1,Raw,forward_ret_5d,2823.0,0.018740,0.212126,0.088342,4.693757,0.545165
2,Sector neutral,forward_ret_1d,2827.0,0.000906,0.158258,0.005728,0.304543,0.503360
3,Sector neutral,forward_ret_5d,2823.0,0.013170,0.156994,0.083889,4.457160,0.539497


### Subperiod stability

In [12]:
periods = {
    "2015-2018": ("2015-01-01", "2018-12-31"),
    "2019-2022": ("2019-01-01", "2022-12-31"),
    "2023-present": ("2023-01-01", "2026-12-31"),
}

subperiod_results = []

for signal_name, factor_column in signal_definitions.items():
    result = calculate_subperiod_ic(
        factor_panel,
        factor_column=factor_column,
        forward_return_column="forward_ret_5d",
        periods=periods,
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    subperiod_results.append(result)

idio_subperiod_summary = pd.concat(
    subperiod_results,
    ignore_index=True,
)

idio_subperiod_summary[
    [
        "signal",
        "period",
        "count",
        "mean_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,signal,period,count,mean_ic,ic_ir,t_stat,positive_fraction
0,Raw,2015-2018,943.0,0.022268,0.124359,3.818860,0.570520
1,Raw,2019-2022,1008.0,0.004877,0.020565,0.652906,0.518849
2,Raw,2023-present,872.0,0.030949,0.144765,4.274863,0.548165
3,Sector neutral,2015-2018,943.0,0.008524,0.056244,1.727165,0.532344
4,Sector neutral,2019-2022,1008.0,0.004327,0.025060,0.795620,0.520833
5,Sector neutral,2023-present,872.0,0.028417,0.199933,5.903950,0.568807


### Redundancy with existing signals

In [13]:
comparison_columns = [
    "idio_vol_63_z",
    "realised_vol_63_z",
    "realised_vol_63_sector_neutral_z",
    "beta_126",
    "mom_12_1m_z",
]

correlation_rows = []

for date, group in factor_panel.groupby("date"):
    valid_idio = group["idio_vol_63_z"].notna().sum()

    if valid_idio < 30:
        continue

    correlations = group[comparison_columns].corr(
        method="spearman"
    ).loc["idio_vol_63_z"]

    correlation_rows.append(
        {
            "date": date,
            "realised_vol": correlations["realised_vol_63_z"],
            "sector_neutral_realised_vol": correlations[
                "realised_vol_63_sector_neutral_z"
            ],
            "beta_126": correlations["beta_126"],
            "mom_12_1m": correlations["mom_12_1m_z"],
        }
    )

daily_factor_correlations = pd.DataFrame(correlation_rows)

correlation_summary = (
    daily_factor_correlations
    .drop(columns="date")
    .agg(["count", "mean", "median", "std", "min", "max"])
    .T
)

correlation_summary

,count,mean,median,std,min,max
realised_vol,2828.0,0.874786,0.918087,0.105341,0.443396,0.989217
sector_neutral_realised_vol,2828.0,0.749747,0.759789,0.084362,0.465101,0.921945
beta_126,2828.0,0.407714,0.417907,0.127536,0.031441,0.678009
mom_12_1m,2639.0,-0.033538,-0.042066,0.224899,-0.545962,0.662886


### Quintile shape

In [14]:
quantile_summary_rows = []

for signal_name, factor_column in signal_definitions.items():
    quantile_returns = calculate_quantile_returns(
        factor_panel,
        factor_column=factor_column,
        forward_return_column="forward_ret_5d",
        quantiles=5,
        min_observations=30,
    )

    averages = (
        quantile_returns
        .groupby("quantile")["mean_forward_return"]
        .mean()
    )

    row = {"signal": signal_name}

    for quantile, value in averages.items():
        row[f"Q{quantile}"] = value

    row["Q5_minus_Q1"] = averages.loc[5] - averages.loc[1]
    quantile_summary_rows.append(row)

idio_quantile_summary = (
    pd.DataFrame(quantile_summary_rows)
    .set_index("signal")
)

idio_quantile_summary

,Q1,Q2,Q3,Q4,Q5,Q5_minus_Q1
signal,,,,,,
Raw,0.002804,0.002697,0.002599,0.004195,0.005672,0.002868
Sector neutral,0.002716,0.002790,0.003572,0.003523,0.005531,0.002815


### Non-overlapping IC

In [15]:
from alpha_research.validation import calculate_non_overlapping_ic

non_overlapping_signals = {
    "Idio vol — raw": "idio_vol_63_z",
    "Idio vol — sector neutral": "idio_vol_63_sector_neutral_z",
    "Realised vol — raw": "realised_vol_63_z",
    "Realised vol — sector neutral": (
        "realised_vol_63_sector_neutral_z"
    ),
}

non_overlapping_results = []

for signal_name, factor_column in non_overlapping_signals.items():
    result = calculate_non_overlapping_ic(
        panel=factor_panel,
        factor_column=factor_column,
        forward_return_column="forward_ret_5d",
        horizon=5,
        method="spearman",
        min_observations=30,
    )

    result["signal"] = signal_name
    non_overlapping_results.append(result)

idio_non_overlapping_ic = pd.concat(
    non_overlapping_results,
    ignore_index=True,
)

idio_non_overlapping_ic[
    [
        "signal",
        "offset",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
].sort_values(["signal", "offset"])

,signal,offset,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,Idio vol — raw,0,565.0,0.016586,0.217364,0.076307,1.813805,0.543363
1,Idio vol — raw,1,564.0,0.020845,0.213912,0.097446,2.314217,0.556738
2,Idio vol — raw,2,564.0,0.017289,0.211750,0.081649,1.939058,0.546099
3,Idio vol — raw,3,565.0,0.020801,0.208024,0.099994,2.376828,0.536283
4,Idio vol — raw,4,565.0,0.018177,0.210178,0.086485,2.055736,0.543363
5,Idio vol — sector neutral,0,565.0,0.010696,0.161227,0.066340,1.576877,0.536283
6,Idio vol — sector neutral,1,564.0,0.014934,0.157372,0.094894,2.253604,0.556738
7,Idio vol — sector neutral,2,564.0,0.012404,0.154582,0.080244,1.905691,0.528369
8,Idio vol — sector neutral,3,565.0,0.015347,0.153126,0.100228,2.382383,0.545133
9,Idio vol — sector neutral,4,565.0,0.012471,0.159032,0.078416,1.863933,0.530973


In [16]:
idio_non_overlapping_summary = (
    idio_non_overlapping_ic
    .groupby("signal")
    .agg(
        mean_ic=("mean_ic", "mean"),
        min_ic=("mean_ic", "min"),
        max_ic=("mean_ic", "max"),
        mean_t_stat=("t_stat", "mean"),
        min_positive_fraction=(
            "positive_fraction",
            "min",
        ),
        max_positive_fraction=(
            "positive_fraction",
            "max",
        ),
    )
)

idio_non_overlapping_summary

,mean_ic,min_ic,max_ic,mean_t_stat,min_positive_fraction,max_positive_fraction
signal,,,,,,
Idio vol — raw,0.018740,0.016586,0.020845,2.099929,0.536283,0.556738
Idio vol — sector neutral,0.013170,0.010696,0.015347,1.996498,0.528369,0.556738
Realised vol — raw,0.025341,0.021475,0.028710,2.061083,0.538053,0.553191
Realised vol — sector neutral,0.019593,0.017110,0.021686,2.446550,0.536283,0.581560


### Properly aligned redundancy comparison

In [17]:
redundancy_pairs = {
    "Raw idio vs raw realised": (
        "idio_vol_63_z",
        "realised_vol_63_z",
    ),
    "Sector-neutral idio vs sector-neutral realised": (
        "idio_vol_63_sector_neutral_z",
        "realised_vol_63_sector_neutral_z",
    ),
    "Raw idio vs beta": (
        "idio_vol_63_z",
        "beta_126",
    ),
    "Sector-neutral idio vs momentum": (
        "idio_vol_63_sector_neutral_z",
        "mom_12_1m_sector_neutral_z",
    ),
}

pairwise_rows = []

for date, group in factor_panel.groupby("date"):
    for pair_name, (left_column, right_column) in (
        redundancy_pairs.items()
    ):
        valid = group[[left_column, right_column]].dropna()

        if len(valid) < 30:
            continue

        pairwise_rows.append(
            {
                "date": date,
                "pair": pair_name,
                "correlation": valid[left_column].corr(
                    valid[right_column],
                    method="spearman",
                ),
            }
        )

daily_pairwise_correlations = pd.DataFrame(pairwise_rows)

aligned_redundancy_summary = (
    daily_pairwise_correlations
    .groupby("pair")["correlation"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

aligned_redundancy_summary

,count,mean,median,std,min,max
pair,,,,,,
Raw idio vs beta,2828,0.407714,0.417907,0.127536,0.031441,0.678009
Raw idio vs raw realised,2828,0.874786,0.918087,0.105341,0.443396,0.989217
Sector-neutral idio vs momentum,2639,-0.033815,-0.025785,0.235958,-0.553188,0.660736
Sector-neutral idio vs sector-neutral realised,2828,0.901492,0.930246,0.085587,0.521552,0.991446


### Transaction-cost and rebalance-offset test

In [18]:
from alpha_research.backtest import (
    BacktestConfig,
    run_rebalance_offset_backtests,
)

idio_backtest_signals = {
    "Idio vol — raw": "idio_vol_63_z",
    "Idio vol — sector neutral": (
        "idio_vol_63_sector_neutral_z"
    ),
    "Realised vol — raw": "realised_vol_63_z",
    "Realised vol — sector neutral": (
        "realised_vol_63_sector_neutral_z"
    ),
}

base_config = BacktestConfig(
    rebalance_frequency=5,
    quantiles=5,
    long_quantile=5,
    short_quantile=1,
    long_gross=1.0,
    short_gross=1.0,
    transaction_cost_bps=10.0,
    min_observations=30,
    rebalance_offset=0,
)

offset_results = []

for signal_name, factor_column in idio_backtest_signals.items():
    result = run_rebalance_offset_backtests(
        panel=factor_panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        base_config=base_config,
    )

    result["signal"] = signal_name
    offset_results.append(result)

idio_offset_backtests = pd.concat(
    offset_results,
    ignore_index=True,
)

idio_offset_backtests[
    [
        "signal",
        "offset",
        "annualised_return",
        "annualised_volatility",
        "sharpe_ratio",
        "max_drawdown",
        "average_rebalance_turnover",
        "total_transaction_cost",
        "maximum_missing_return_weight",
    ]
].sort_values(["signal", "offset"])

,signal,offset,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown,average_rebalance_turnover,total_transaction_cost,maximum_missing_return_weight
0,Idio vol — raw,0,0.103869,0.181682,0.634933,-0.396790,0.400173,0.231300,0.0
1,Idio vol — raw,1,0.101314,0.180777,0.624407,-0.361038,0.391522,0.226300,0.0
2,Idio vol — raw,2,0.102956,0.180713,0.632814,-0.347621,0.394637,0.228100,0.0
3,Idio vol — raw,3,0.110770,0.181259,0.670358,-0.357987,0.399135,0.230700,0.0
4,Idio vol — raw,4,0.116470,0.180523,0.700739,-0.360968,0.404152,0.233600,0.0
5,Idio vol — sector neutral,0,0.101846,0.126813,0.828321,-0.210009,0.473957,0.273947,0.0
6,Idio vol — sector neutral,1,0.098701,0.126152,0.809330,-0.208422,0.477636,0.276074,0.0
7,Idio vol — sector neutral,2,0.111815,0.127113,0.897551,-0.197629,0.471517,0.272537,0.0
8,Idio vol — sector neutral,3,0.116427,0.127753,0.926100,-0.230975,0.469140,0.271163,0.0
9,Idio vol — sector neutral,4,0.125976,0.126675,1.000167,-0.212123,0.472382,0.273037,0.0


### Summarise robustness

In [19]:
idio_backtest_summary = (
    idio_offset_backtests
    .groupby("signal")
    .agg(
        mean_annualised_return=(
            "annualised_return",
            "mean",
        ),
        min_annualised_return=(
            "annualised_return",
            "min",
        ),
        max_annualised_return=(
            "annualised_return",
            "max",
        ),
        mean_sharpe=("sharpe_ratio", "mean"),
        min_sharpe=("sharpe_ratio", "min"),
        max_sharpe=("sharpe_ratio", "max"),
        mean_max_drawdown=("max_drawdown", "mean"),
        mean_rebalance_turnover=(
            "average_rebalance_turnover",
            "mean",
        ),
    )
)

idio_backtest_summary

,mean_annualised_return,min_annualised_return,max_annualised_return,mean_sharpe,min_sharpe,max_sharpe,mean_max_drawdown,mean_rebalance_turnover
signal,,,,,,,,
Idio vol — raw,0.107076,0.101314,0.116470,0.652650,0.624407,0.700739,-0.364881,0.397924
Idio vol — sector neutral,0.110953,0.098701,0.125976,0.892294,0.809330,1.000167,-0.211832,0.472927
Realised vol — raw,0.159706,0.157869,0.162929,0.737234,0.731003,0.746151,-0.436808,0.348997
Realised vol — sector neutral,0.117574,0.111268,0.126245,0.796492,0.763370,0.844509,-0.279367,0.423504


## Test residualised idio vol signal

### Construct the daily residualised signal

In [20]:
def add_cross_sectional_residualised_signal(
    panel,
    target_column,
    control_column,
    output_column,
    min_observations=30,
):
    result = panel.copy()
    result[output_column] = np.nan

    for _, indices in result.groupby("date").groups.items():
        valid = result.loc[
            indices,
            [target_column, control_column],
        ].dropna()

        if len(valid) < min_observations:
            continue

        y = valid[target_column].to_numpy()
        x = valid[control_column].to_numpy()

        design_matrix = np.column_stack(
            [np.ones(len(valid)), x]
        )

        coefficients, *_ = np.linalg.lstsq(
            design_matrix,
            y,
            rcond=None,
        )

        residuals = y - design_matrix @ coefficients
        residual_std = residuals.std(ddof=1)

        if not np.isfinite(residual_std) or residual_std == 0:
            continue

        result.loc[valid.index, output_column] = (
            residuals - residuals.mean()
        ) / residual_std

    return result


factor_panel = add_cross_sectional_residualised_signal(
    panel=factor_panel,
    target_column="idio_vol_63_sector_neutral_z",
    control_column="realised_vol_63_sector_neutral_z",
    output_column="idio_vol_residualised_z",
    min_observations=30,
)

### Verify that realised-vol exposure was removed

In [21]:
residualisation_diagnostics = []

for date, group in factor_panel.groupby("date"):
    valid = group[
        [
            "idio_vol_residualised_z",
            "realised_vol_63_sector_neutral_z",
        ]
    ].dropna()

    if len(valid) < 30:
        continue

    residualisation_diagnostics.append(
        {
            "date": date,
            "pearson_correlation": valid[
                "idio_vol_residualised_z"
            ].corr(
                valid[
                    "realised_vol_63_sector_neutral_z"
                ],
                method="pearson",
            ),
            "spearman_correlation": valid[
                "idio_vol_residualised_z"
            ].corr(
                valid[
                    "realised_vol_63_sector_neutral_z"
                ],
                method="spearman",
            ),
        }
    )

residualisation_diagnostics = pd.DataFrame(
    residualisation_diagnostics
)

residualisation_diagnostics[
    ["pearson_correlation", "spearman_correlation"]
].agg(["count", "mean", "median", "std", "min", "max"])

,pearson_correlation,spearman_correlation
count,2.828000e+03,2828.000000
mean,-4.736400e-18,0.020444
median,-1.177578e-17,0.019122
std,9.250489e-16,0.049550
min,-3.736660e-15,-0.144197
max,4.028491e-15,0.223915


### Non-overlapping incremental IC

In [22]:
residualised_ic = calculate_non_overlapping_ic(
    panel=factor_panel,
    factor_column="idio_vol_residualised_z",
    forward_return_column="forward_ret_5d",
    horizon=5,
    method="spearman",
    min_observations=30,
)

residualised_ic[
    [
        "offset",
        "count",
        "mean_ic",
        "std_ic",
        "ic_ir",
        "t_stat",
        "positive_fraction",
    ]
]

,offset,count,mean_ic,std_ic,ic_ir,t_stat,positive_fraction
0,0,565.0,-0.010191,0.147555,-0.069069,-1.641747,0.483186
1,1,564.0,-0.011769,0.142288,-0.082714,-1.964344,0.484043
2,2,564.0,-0.011353,0.145877,-0.077828,-1.848312,0.462766
3,3,565.0,-0.009102,0.144945,-0.062793,-1.492582,0.460177
4,4,565.0,-0.010556,0.149584,-0.070571,-1.677463,0.465487


In [28]:
residualised_ic_summary = residualised_ic.agg(
    mean_ic=("mean_ic", "mean"),
    min_ic=("mean_ic", "min"),
    max_ic=("mean_ic", "max"),
    mean_t_stat=("t_stat", "mean"),
    min_positive_fraction=("positive_fraction", "min"),
    max_positive_fraction=("positive_fraction", "max"),
).T.bfill().iloc[[0]].reset_index(drop=True)

residualised_ic_summary

,mean_ic,min_ic,max_ic,mean_t_stat,min_positive_fraction,max_positive_fraction
0,-0.010594,-0.011769,-0.009102,-1.72489,0.460177,0.484043


### Tradability after residualisation

In [29]:
residualised_offset_backtests = (
    run_rebalance_offset_backtests(
        panel=factor_panel,
        factor_column="idio_vol_residualised_z",
        return_column="forward_ret_1d",
        base_config=base_config,
    )
)

residualised_offset_backtests[
    [
        "offset",
        "annualised_return",
        "annualised_volatility",
        "sharpe_ratio",
        "max_drawdown",
        "average_rebalance_turnover",
        "total_transaction_cost",
        "maximum_missing_return_weight",
    ]
]

,offset,annualised_return,annualised_volatility,sharpe_ratio,max_drawdown,average_rebalance_turnover,total_transaction_cost,maximum_missing_return_weight
0,0,-0.058168,0.113133,-0.473099,-0.556956,0.647860,0.374463,0.0
1,1,-0.062392,0.113153,-0.512712,-0.576758,0.653715,0.377847,0.0
2,2,-0.062468,0.115216,-0.502196,-0.595204,0.649645,0.375495,0.0
3,3,-0.053852,0.114584,-0.425763,-0.545204,0.653069,0.377474,0.0
4,4,-0.066055,0.113777,-0.543671,-0.614631,0.655336,0.378784,0.0


In [31]:
residualised_backtest_summary = (
    residualised_offset_backtests.agg(
        mean_annualised_return=(
            "annualised_return",
            "mean",
        ),
        min_annualised_return=(
            "annualised_return",
            "min",
        ),
        max_annualised_return=(
            "annualised_return",
            "max",
        ),
        mean_sharpe=("sharpe_ratio", "mean"),
        min_sharpe=("sharpe_ratio", "min"),
        max_sharpe=("sharpe_ratio", "max"),
        mean_max_drawdown=("max_drawdown", "mean"),
        mean_rebalance_turnover=(
            "average_rebalance_turnover",
            "mean",
        ),
    )
).T.bfill().iloc[[0]].reset_index(drop=True)

residualised_backtest_summary

,mean_annualised_return,min_annualised_return,max_annualised_return,mean_sharpe,min_sharpe,max_sharpe,mean_max_drawdown,mean_rebalance_turnover
0,-0.060587,-0.066055,-0.053852,-0.491488,-0.543671,-0.425763,-0.577751,0.651925


## Conclusion

This experiment implemented and validated a 63-day idiosyncratic-volatility factor using residual volatility from a rolling stock–SPY market model.

### Main findings

- **Positive standalone signal:** Higher idiosyncratic volatility was associated with higher subsequent returns in this S&P 100 universe.
- **Robustness:** The positive 5-day IC survived non-overlapping sampling across all five offsets:
  - Raw idio vol: mean IC **1.87%**
  - Sector-neutral idio vol: mean IC **1.32%**
- **Tradability:** Performance remained positive after 10 bps per-side transaction costs and was stable across rebalance offsets.
- **Best implementation:** Sector-neutral idio vol produced the strongest risk-adjusted results:
  - Mean annualised return: **11.1%**
  - Mean Sharpe ratio: **0.89**
  - Mean maximum drawdown: **−21.2%**
- **High redundancy:** Sector-neutral idio vol had an average cross-sectional correlation of approximately **0.90** with sector-neutral realised volatility.
- **No positive incremental effect:** After residualising idio vol against realised vol, the remaining component produced:
  - Mean IC: **−1.06%**
  - Mean annualised return for the long-high strategy: **−6.1%**
  - Mean Sharpe ratio: **−0.49**

### Research decision

Idiosyncratic volatility is a **validated alternative volatility specification**, but not an independent incremental factor. Its successful standalone performance is largely explained by the component shared with realised volatility.

For the current research pipeline:

- retain idio vol in the factor panel as a researched alternative;
- continue using **sector-neutral realised volatility** as the primary volatility factor;
- do not combine positive realised vol and positive idio vol as separate signals, since this would largely double-count the same effect;
- treat the reversed residualised signal only as a possible future experiment, because its direction was discovered in-sample and it has relatively high turnover.

Overall, this experiment demonstrates that a more refined factor definition does not necessarily provide independent information. Testing redundancy and residualised performance was essential for distinguishing a strong standalone backtest from genuine incremental alpha.